# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

# Display high-level information about the dataset
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Dataset ID: {metadata['@id']}")

# Display sample keywords and other relevant metadata
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset contains multiple structured tables (record sets) with clinical features, hormone levels, and genetic variants.

Let's enumerate available record sets as referenced by their `@id` fields. Then, for each record set, show the field and column `@id`s.

In [ ]:
# List all record sets by their @id
record_sets = dataset.record_sets

print("Available record sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For each record set, print fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    fields = rs.get('fields', [])
    print(f"Fields:")
    for field in fields:
        print(f"  - {field['@id']} (name: {field.get('name', 'N/A')})")
        columns = field.get('columns', [])
        if columns:
            print("    Columns:")
            for col in columns:
                print(f"      * {col['@id']} (name: {col.get('name', 'N/A')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Each record set, field, and column is referenced by its `@id`.

We'll extract all available record sets, review their columns, and preview the data.

In [ ]:
# Collect @id for all available record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load records for each record set by @id
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns and a preview for each record set
for rs_id in record_set_ids:
    print(f"\nRecordSet {rs_id} contains columns:")
    print(dataframes[rs_id].columns.tolist())
    print(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll apply filtering, normalization, and grouping operations, referencing columns by their `@id`.

For demonstration, let's:
- Filter patients by age (using the age column `@id` from Table 1 record set),
- Normalize age,
- Group by presence of infertility (using its `@id` column).

Replace these `@id`s with those provided by the schema for correct execution.

In [ ]:
# Choose a record set representing clinical features (e.g., Table 1)
# Replace with actual @id from your schema, for demonstration we'll use placeholders:
clinical_rs_id = record_set_ids[0]  # May need to adjust order; check overview
clinical_df = dataframes[clinical_rs_id]

# Replace these with actual @id from overview
age_id = None
infertility_id = None

# Find candidate field names matching age and infertility
for c in clinical_df.columns:
    if 'age' in c.lower():
        age_id = c
    if 'infertility' in c.lower():
        infertility_id = c

if age_id is None or infertility_id is None:
    print("Couldn't automatically identify age or infertility column @id. Please refer to record set overview.")
else:
    # Filter records by age > 20
    threshold = 20
    filtered_df = clinical_df[clinical_df[age_id] > threshold]
    print(f"Filtered records with {age_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize age
    filtered_df[f"{age_id}_normalized"] = (filtered_df[age_id] - filtered_df[age_id].mean()) / filtered_df[age_id].std()
    print(f"Normalized {age_id} for filtered records:")
    print(filtered_df[[age_id, f"{age_id}_normalized"]].head())

    # Group by infertility
    if infertility_id in clinical_df.columns:
        grouped_df = filtered_df.groupby(infertility_id).mean(numeric_only=True)
        print(f"Grouped data by {infertility_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields, referencing columns by their `@id`.

We'll plot:
- Distribution of age,
- Relationship between age and hormone levels.

Remember to substitute `@id` for hormone columns based on the schema overview.

In [ ]:
import matplotlib.pyplot as plt

# Use previously identified age column
if age_id is not None:
    plt.figure(figsize=(6,4))
    plt.hist(clinical_df[age_id].dropna(), bins=5, color='skyblue', edgecolor='black')
    plt.title('Age Distribution')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

# Identify hormone level column from other record sets
hormone_rs_id = None
hormone_col_id = None
# Try to automatically find a record set that contains 'hormone' in columns
for rs_id in record_set_ids:
    df = dataframes[rs_id]
    for c in df.columns:
        if 'hormone' in c.lower():
            hormone_rs_id = rs_id
            hormone_col_id = c
            break
    if hormone_rs_id:
        break

if hormone_rs_id and hormone_col_id and age_id in clinical_df.columns:
    # Merge on patient identifier (replace 'patient_id' with actual column @id as needed)
    common_col = None
    for c in clinical_df.columns:
        if 'id' in c.lower():
            common_col = c
    for c in dataframes[hormone_rs_id].columns:
        if 'id' in c.lower():
            hormone_common_col = c
    if common_col and hormone_common_col:
        merged_df = pd.merge(clinical_df, dataframes[hormone_rs_id], left_on=common_col, right_on=hormone_common_col, how='inner')
        plt.figure(figsize=(6,4))
        plt.scatter(merged_df[age_id], merged_df[hormone_col_id], color='purple')
        plt.title(f'Age vs Hormone Level ({hormone_col_id})')
        plt.xlabel('Age')
        plt.ylabel(hormone_col_id)
        plt.show()
    else:
        print("Couldn't find a suitable patient identifier column for merge.")
else:
    print("No hormone columns found for visualization.")

## 6. Conclusion
This notebook demonstrated a guided exploration of the FAIR^2 dataset using `mlcroissant`. All entities—record sets, fields, columns—were referenced using their `@id` for reproducibility and schema-consistency, enabling metadata-driven tabulation, filtering, normalization, and visualization.

Key findings include:
- Successful extraction and preview of all record sets.
- Basic filtering and normalization applied to clinical features.
- Data grouped by key clinical outcome.
- Visualizations of patient age and hormone distributions.

For further analysis or ML development, always refer to the Croissant schema `@id` for precise feature selection and reproducibility.